# 🧱 Multi-Layer Perceptron (MLP) — Solutions Notebook

**This notebook contains complete, verified solutions.**

**Difficulty**: ⭐ Beginner  
**Time**: ~45 minutes

---


## 🎯 Section 1: Overview

A **Multi-Layer Perceptron (MLP)** is a class of feedforward artificial neural network. It consists of an input layer, one or more hidden layers, and an output layer. Except for the input nodes, each node is a neuron that uses a non-linear activation function. MLP utilizes backpropagation for training the network.

### Applications
- Tabular data classification and regression
- Simple image classification (e.g. MNIST digit classification)
- Function approximation


## 📐 Section 2: Math & Intuition

### Layer Equations
For a layer $l$ with weights $W^{[l]}$, bias $b^{[l]}$, and activation function $g^{[l]}$:
$$Z^{[l]} = A^{[l-1]} W^{[l]} + b^{[l]}$$
$$A^{[l]} = g^{[l]}(Z^{[l]})$$
where $A^{[0]} = X$ (input data).

### Activations
- **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$ with derivative $\sigma'(z) = \sigma(z)(1 - \sigma(z))$
- **ReLU**: $\text{ReLU}(z) = \max(0, z)$ with derivative $\text{ReLU}'(z) = \mathbb{1}(z > 0)$

### Binary Cross-Entropy Loss
$$J = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \log(a_2^{(i)}) + (1 - y^{(i)}) \log(1 - a_2^{(i)}) \right]$$

### Backpropagation (2-layer MLP)
We want to compute gradients of cost $J$ with respect to parameters $W^{[2]}, b^{[2]}, W^{[1]}, b^{[1]}$:
- Output layer error: $dZ^{[2]} = A^{[2]} - Y$ (assuming sigmoid activation + binary cross-entropy loss)
- Gradients: 
  $$dW^{[2]} = \frac{1}{m} (A^{[1]})^T dZ^{[2]}$$
  $$db^{[2]} = \frac{1}{m} \sum dZ^{[2]}$$
- Hidden layer error:
  $$dZ^{[1]} = (dZ^{[2]} (W^{[2]})^T) * g^{[1]}' (Z^{[1]})$$
- Gradients:
  $$dW^{[1]} = \frac{1}{m} X^T dZ^{[1]}$$
  $$db^{[1]} = \frac{1}{m} \sum dZ^{[1]}$$


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('MLP Setup complete! ✅')


### 3.1 Generate Synthetic Non-linear Data


In [ ]:
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
y = y.reshape(-1, 1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[y.ravel()==0, 0], X[y.ravel()==0, 1], label='Class 0', alpha=0.8)
plt.scatter(X[y.ravel()==1, 0], X[y.ravel()==1, 1], label='Class 1', alpha=0.8)
plt.title('Synthetic Moons Dataset')
plt.legend()
plt.show()


### 3.2 MLP Implementation


In [ ]:
class MLPFromScratch:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.1):
        self.lr = lr
        # Initialize weights randomly, biases to zero
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.1
        self.b2 = np.zeros((1, output_dim))
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, a):
        # Note: input is already activated 'a'
        return a * (1 - a)

    def relu(self, z):
        return np.maximum(0, z)

    def relu_derivative(self, z):
        # Note: input is pre-activated 'z'
        return (z > 0).astype(float)

    def forward(self, X):
        # TODO: Implement forward pass. Return cache dict.
        Z1 = X @ self.W1 + self.b1
        A1 = self.relu(Z1)
        Z2 = A1 @ self.W2 + self.b2
        A2 = self.sigmoid(Z2)
        return {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}

    def backward(self, X, y, cache):
        # TODO: Implement backward pass. Return gradients dict.
        m = X.shape[0]
        A1 = cache['A1']
        A2 = cache['A2']
        Z1 = cache['Z1']
        
        dZ2 = A2 - y
        dW2 = (A1.T @ dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m
        
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_derivative(Z1)
        dW1 = (X.T @ dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m
        
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}

    def update_params(self, grads):
        self.W1 -= self.lr * grads['dW1']
        self.b1 -= self.lr * grads['db1']
        self.W2 -= self.lr * grads['dW2']
        self.b2 -= self.lr * grads['db2']

    def compute_loss(self, y, a2):
        # Binary cross entropy loss with clipping to avoid log(0)
        a2 = np.clip(a2, 1e-15, 1 - 1e-15)
        return -np.mean(y * np.log(a2) + (1 - y) * np.log(1 - a2))
        
    def fit(self, X, y, epochs=1000):
        history = []
        for epoch in range(epochs):
            cache = self.forward(X)
            loss = self.compute_loss(y, cache['A2'])
            grads = self.backward(X, y, cache)
            self.update_params(grads)
            history.append(loss)
        return history
        
    def predict(self, X):
        cache = self.forward(X)
        return (cache['A2'] >= 0.5).astype(int)


### 3.3 Verify Implementation


In [ ]:
mlp = MLPFromScratch(input_dim=2, hidden_dim=8, output_dim=1, lr=0.1)
if 'TODO' not in mlp.forward.__code__.co_consts:
    loss_history = mlp.fit(X_train, y_train, epochs=2000)
    preds = mlp.predict(X_test)
    acc = np.mean(preds == y_test)
    print(f'Training Final Loss: {loss_history[-1]:.4f}')
    print(f'Test Accuracy: {acc * 100:.2f}%')
    assert acc >= 0.8, 'Accuracy should be at least 80%'
    
    # Plot loss
    plt.figure(figsize=(6, 4))
    plt.plot(loss_history)
    plt.title('NumPy MLP Convergence')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.show()
else:
    print('Skipping test - class functions not yet implemented')


## 📦 Section 4: Library Implementation


Now let's build the equivalent MLP using PyTorch. We will use `nn.Module` to define our model structure, `optim.Adam` to optimize parameters, and write the training loop.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PyTorchMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)



### 4.1 PyTorch Training Loop


In [ ]:
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

torch.manual_seed(42)
model = PyTorchMLP(input_dim=2, hidden_dim=8, output_dim=1)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.1)

# Solution training loop
epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_outputs = model(X_test_t)
    test_preds = (test_outputs >= 0.5).float()
    test_acc = (test_preds == y_test_t).float().mean().item()
    print(f'PyTorch Test Accuracy: {test_acc * 100:.2f}%')
    assert test_acc >= 0.8, 'PyTorch accuracy should be at least 80%'


## 🧪 Section 5: Experiments


Compare the impact of different activation functions on training convergence.


In [ ]:
# Experiment: ReLU vs Sigmoid vs Tanh on MLP
class PyTorchFlexMLP(nn.Module):
    def __init__(self, act_fn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            act_fn,
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

activations = {
    'ReLU': nn.ReLU(),
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh()
}

plt.figure(figsize=(10, 6))
for name, act in activations.items():
    torch.manual_seed(42)
    m = PyTorchFlexMLP(act)
    opt = optim.Adam(m.parameters(), lr=0.01)
    crit = nn.BCELoss()
    losses = []
    for epoch in range(100):
        opt.zero_grad()
        out = m(X_train_t)
        l = crit(out, y_train_t)
        l.backward()
        opt.step()
        losses.append(l.item())
    plt.plot(losses, label=name)

plt.title('Effect of Activation Functions on Convergence')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


## ❓ Section 6: Interview Questions


### Q1: Explain backpropagation and its reliance on the chain rule.
**Answer**:
Backpropagation is the algorithm used to calculate parameters' gradients in neural networks. It works by computing the gradient of the loss function with respect to each weight and bias by applying the chain rule of calculus. The computation starts at the final layer (output) and propagates backwards through the network layers to calculate gradients layer-by-layer.

### Q2: What is the Vanishing Gradient problem? How do activation functions like ReLU help mitigate it?
**Answer**:
The vanishing gradient problem occurs when gradients of the loss function approach zero as they are backpropagated through many layers, causing the early layers to update very slowly (or not at all). Activation functions like Sigmoid and Tanh saturate for very large positive or negative values, meaning their derivatives go to zero. ReLU mitigates this because its derivative is constant ($1$) for all positive inputs, allowing gradient flow to remain strong through deep layers.

### Q3: What is a 'dead ReLU' and how do you prevent it?
**Answer**:
A 'dead ReLU' occurs when a neuron gets stuck in the inactive state (outputting $0$) because the input to it is always negative, leading to a gradient of $0$ during backpropagation. Since the gradient is zero, the weights will never update, and the neuron remains 'dead'. This can be prevented by: using Leaky ReLU (which has a small non-zero slope for negative inputs), lower learning rates, or proper weight initialization (such as He initialization).

### Q4: Explain the difference between Xavier (Glorot) and He (Kaiming) initialization.
**Answer**:
- **Xavier Glorot Initialization** is designed for symmetric activation functions (like Sigmoid and Tanh). It sets weights drawn from a distribution with variance $\text{Var}(W) = \frac{2}{N_{\text{in}} + N_{\text{out}}}$.
- **He Kaiming Initialization** is designed specifically for non-symmetric, rectified activations (like ReLU). Since ReLU discards half the input variance (inputs $<0$), He initialization accounts for this by drawing weights with variance $\text{Var}(W) = \frac{2}{N_{\text{in}}}$, scaling the weights larger to maintain signal strength.

### Q5: Why do we need non-linear activation functions in deep networks?
**Answer**:
Without non-linear activation functions, a multi-layer neural network would simply represent a composition of linear transformations. Because a composition of linear transformations is mathematically equivalent to a single linear transformation (i.e. $W_2(W_1 x + b_1) + b_2 = W_{new} x + b_{new}$), a network of any depth without non-linearities would only be able to learn linear decision boundaries, defeating the purpose of deep representations.


## 🏆 Section 7: Challenge — Multiclass MLP


**Challenge**: Implement the Softmax activation function and Categorical Cross-Entropy Loss from scratch, and test it on a multi-class dataset.


In [ ]:
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
def categorical_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]
    
# Simple check
z_test = np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]])
s = softmax(z_test)
print('Softmax output (sums to 1 per row):\n', s)
assert np.allclose(s.sum(axis=1), 1.0)

y_t = np.array([[0, 0, 1], [1, 0, 0]])
loss_val = categorical_cross_entropy(y_t, s)
print('Categorical CE Loss:', loss_val)
assert loss_val > 0
